# Aumentación de Datos en la Iteración 3b (Estudio de Ablación)
###1. ¿Qué se hizo?
Se diseñó un estudio de ablación controlada manteniendo exactamente la misma arquitectura (xlm-roberta-large), la misma función de pérdida sin suavizado (CrossEntropyLoss()), los mismos hiperparámetros (LR=1e-5, EPOCHS=4, BATCH_SIZE=16) y la misma partición de validación estratificada (seed=42) que en la Iteración 2 y 3, pero incorporando 1,944 ejemplos sintéticos generados exclusivamente sobre el conjunto de entrenamiento (trn_df, pasando de 17,055 a 18,999 registros) mediante dos técnicas intra-reseña de bajo ruido:
* Code-Switching Léxico Intra-Reseña (aug_lexical_cs, 544 ejemplos): Sobre reseñas originales en español o inglés, se sustituyeron hasta dos términos o locuciones neutras de dominio (como "la batería" $\leftrightarrow$ "the battery", "el envío" $\leftrightarrow$ "the shipping", "calidad" $\leftrightarrow$ "quality", "en la oficina" $\leftrightarrow$ "at the office") manteniendo intacto el resto de la oración y la etiqueta original.Word Dropout
* Protegido en Reseñas con Concesiones (aug_protected_dropout, 1,400 ejemplos): Se filtraron reseñas que contienen conectores adversativos (pero, aunque, sin embargo, but, however, mas, porém, mais) y se eliminó una única palabra descriptiva larga ($>4$ caracteres), protegiendo explícitamente una lista blanca (PROTECTED) de adverbios de negación (no, nunca, tampoco, not, never, não, pas) y conectores de contraste.
###2. ¿Por qué se decidió hacer eso?
Aislar el efecto real de la aumentación de datos: En la Iteración 3 se habían modificado simultáneamente tres variables (traducción automática MarianMT, mezcla de oraciones entre reseñas distintas y label_smoothing=0.03), lo que impidió saber cuál de esos cambios causó la caída en el leaderboard público de Kaggle (0.89685 $\rightarrow$ 0.89047). La Iteración 4 fijó todas las demás variables para medir únicamente el impacto limpio de la aumentación.   Preservar la coherencia temática y la estructura sintáctica: A diferencia de concatenar oraciones de dos reseñas distintas (que mezclaba productos diferentes en un mismo párrafo), el reemplazo léxico puntual y la eliminación de una palabra descriptiva conservan el hilo conductor de la opinión del comprador y respetan las negaciones de largo alcance.   

Prevenir fuga de datos (data leakage): Toda la generación sintética se aplicó estrictamente después de separar trn_df y val_df, garantizando que ninguna variante aumentada de una reseña de validación apareciera en entrenamiento.  
###3. ¿Qué otras alternativas había y por qué no se hicieron?
* Traducción automática neuronal en una sola dirección (MarianMT es -> en / es -> fr): Ya se había probado en la Iteración 3 y se descartó aquí porque los traductores ligeros tienden a neutralizar modismos e ironías, además de alterar las proporciones de idiomas en entrenamiento y desplazar la frontera de decisión hacia la clase positivo (45.80% en eval.csv).
* Mezcla de oraciones entre reseñas distintas (Sentence Mixing Code-Switching): Se descartó porque unir fragmentos de dos compradores diferentes genera secuencias artificiales fuera de distribución (out-of-distribution) que degradaron el desempeño en es-en (0.6818 $\rightarrow$ 0.6364).
* Perturbación aleatoria sin diccionario de protección (EDA estándar): Eliminar o intercambiar palabras al azar en reseñas con concesiones o doble negación ("no diría que sea malo") invierte el sentimiento real del texto e introduce etiquetas ruidosas.
* Sobremuestreo aleatorio simple (Random Oversampling): Duplicar filas exactas sin variación léxica no aporta nueva información sintáctica al Transformer y acelera el sobreajuste (overfitting) sobre las muestras repetidas.
###4. ¿Qué cambia frente al código anterior (Iteraciones 2 y 3)?
Frente a la Iteración 2 (xlm-roberta-large sin aumentación): Se pasa de entrenar exclusivamente con los 17,055 ejemplos originales a 18,999 ejemplos (+11.4%), manteniendo idénticos los pesos iniciales, el optimizador AdamW, el scheduler coseno y la pérdida CrossEntropyLoss().Frente a la Iteración 3 (aumentación con traducción y label_smoothing):Se eliminan por completo las dependencias externas de traducción (MarianMTModel, MarianTokenizer) y la mezcla entre reseñas distintas.Se elimina label_smoothing=0.03, volviendo a la entropía cruzada estándar para evitar suavizar en exceso los logits de la frontera negativo vs. positivo.Se reduce el volumen de datos sintéticos (de +4,438 filas en Iteración 3 a +1,944 filas en Iteración 4) priorizando perturbaciones mínimas dentro de la misma reseña.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Dispositivo:", device)

train_df = pd.read_csv('train.csv', encoding='utf-8')
eval_df = pd.read_csv('eval.csv', encoding='utf-8')

label2id = {'negativo': 0, 'neutral': 1, 'positivo': 2}
id2label = {0: 'negativo', 1: 'neutral', 2: 'positivo'}
train_df['target'] = train_df['label'].map(label2id)

def detect_lang(text):
    t = text.lower()
    words = set(re.findall(r'\b[a-záéíóúñãõçâêîôûàèù]+\b', t))
    es = len(words & {'el','la','los','las','que','en','un','una','por','para','con','pero','muy','lo','me','se','del','al','es','son','fue','está','más','también','cuando','todo','esta','este','bien','malo','bueno','llegó','compré','pedí','uso','días','casa'})
    en = len(words & {'the','and','to','of','is','in','it','for','that','with','on','was','but','not','are','have','this','my','very','good','great','bad','product','bought','use','days','quality','battery','would','after'})
    pt = len(words & {'não','com','uma','para','por','mais','como','mas','muito','bem','bom','boa','ruim','comprei','chegou','dias','casa','uso','também','quando','isso','essa','esse','são','está','fazer','qualidade','bateria','nas','dos','das','pelo','pela'})
    fr = len(words & {'le','la','les','des','est','une','dans','pour','pas','que','avec','sur','mais','plus','très','bien','bon','mauvais','acheté','livraison','jours','maison','qualité','batterie','depuis','tout','cette','sont','fait','avoir'})
    if es >= 3 and en >= 3:
        return 'es-en'
    scores = {'es': es, 'en': en, 'pt': pt, 'fr': fr}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'other'

train_df['lang'] = train_df['text'].apply(detect_lang)
eval_df['lang'] = eval_df['text'].apply(detect_lang)

# Mismo split exacto que Iteraciones #2 y #3 (seed=42)
train_df['strat_key'] = train_df['label'] + '_' + train_df['lang']
skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)
for train_idx, val_idx in skf.split(train_df, train_df['strat_key']):
    trn_df = train_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_df.iloc[val_idx].reset_index(drop=True)
    break

# --- ESTRATEGIA DE AUMENTACIÓN LIMPIA (SOLO EN trn_df) ---

# 1. Code-Switching Léxico Intra-Reseña (Reemplazo de términos neutros de dominio ES <-> EN sin alterar polaridad)
CS_PAIRS = [
    (r'\bla batería\b', 'the battery'), (r'\bel teclado\b', 'the keyboard'),
    (r'\blos audífonos\b', 'the headphones'), (r'\bel envío\b', 'the shipping'),
    (r'\bcalidad\b', 'quality'), (r'\bprecio\b', 'price'),
    (r'\ben la oficina\b', 'at the office'), (r'\ba mi casa\b', 'to my house'),
    (r'\bthe battery\b', 'la batería'), (r'\bthe quality\b', 'la calidad'),
    (r'\bthe price\b', 'el precio'), (r'\bshipping\b', 'envío')
]

def aug_lexical_cs(df, n_samples=1500):
    candidates = df[df['lang'].isin(['es', 'en'])].sample(n=n_samples, random_state=42)
    rows = []
    for _, r in candidates.iterrows():
        text = r['text']
        new_text = text
        applied = 0
        for pat, rep in random.sample(CS_PAIRS, len(CS_PAIRS)):
            if re.search(pat, new_text, flags=re.IGNORECASE):
                new_text = re.sub(pat, rep, new_text, count=1, flags=re.IGNORECASE)
                applied += 1
            if applied >= 2:
                break
        if new_text != text:
            rows.append({'text': new_text, 'label': r['label'], 'lang': 'es-en', 'aug_type': 'lexical_cs'})
    return pd.DataFrame(rows)

# 2. Protected Word Dropout (elimina 1 palabra descriptiva larga en reseñas con concesiones, protegiendo negaciones/conectores)
PROTECTED = {
    'no','nunca','tampoco','ni','sin','pero','aunque','embargo','sino','nada','malo','mala','bueno','buena',
    'not','never','no','without','but','although','however','yet','bad','good','great','poor',
    'não','nunca','sem','mas','porém','embora','ruim','bom','boa',
    'pas','jamais','sans','mais','cependant','bien','bon','mauvais'
}

def aug_protected_dropout(df, n_samples=1400):
    mask = df['text'].str.lower().str.contains(r'\b(?:pero|aunque|sin embargo|but|however|mas|porém|mais)\b', regex=True)
    cand = df[mask].sample(n=min(n_samples, mask.sum()), random_state=42)
    rows = []
    for _, r in cand.iterrows():
        words = r['text'].split()
        elig = [i for i, w in enumerate(words) if re.sub(r'\W+', '', w.lower()) not in PROTECTED and len(w) > 4]
        if len(elig) >= 2:
            w_copy = words.copy()
            w_copy.pop(random.choice(elig))
            rows.append({'text': ' '.join(w_copy), 'label': r['label'], 'lang': r['lang'], 'aug_type': 'protected_dropout'})
    return pd.DataFrame(rows)

df_cs = aug_lexical_cs(trn_df, n_samples=1500)
df_dr = aug_protected_dropout(trn_df, n_samples=1400)

trn_df['aug_type'] = 'original'
trn_aug = pd.concat([trn_df[['text', 'label', 'lang', 'aug_type']], df_cs, df_dr], ignore_index=True)
trn_aug['target'] = trn_aug['label'].map(label2id)
trn_aug = trn_aug.sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Resumen datos aumentados (Ablación Limpia):")
print(trn_aug['aug_type'].value_counts())
print(f"Total entrenamiento: {len(trn_df)} -> {len(trn_aug)} | Validación intacta: {len(val_df)}")

Dispositivo: cuda


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=6.
  warnings.warn(


Resumen datos aumentados (Ablación Limpia):
aug_type
original             17055
protected_dropout     1400
lexical_cs             544
Name: count, dtype: int64
Total entrenamiento: 17055 -> 18999 | Validación intacta: 3411


In [ ]:
MODEL_NAME = 'xlm-roberta-large'
MAX_LEN = 192
BATCH_SIZE = 16
EPOCHS = 4
LR = 1e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx]
        }
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

trn_ds = ReviewDataset(trn_aug['text'], trn_aug['target'].values)
val_ds = ReviewDataset(val_df['text'], val_df['target'].values)
eval_ds = ReviewDataset(eval_df['text'])

trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(trn_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in trn_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            # Pérdida CrossEntropy estándar (sin label smoothing) idéntica a Iteración #2
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    model.eval()
    val_probs, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            val_probs.append(probs)
            val_targets.extend(batch['labels'].numpy())

    val_probs = np.vstack(val_probs)
    val_preds = np.argmax(val_probs, axis=1)
    val_acc = accuracy_score(val_targets, val_preds)
    print(f"Época {epoch+1}/{EPOCHS} | Train Loss: {total_loss/len(trn_loader):.4f} | Val Accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_xlmr_large_clean_aug.pt')
        np.save('val_probs_xlmr_large_clean_aug.npy', val_probs)
        val_df['pred_clean_aug'] = val_preds

print(f"\nMejor Val Accuracy (Aug Limpia): {best_val_acc:.4f} (vs Sin Aug: 0.9074 | vs Aug Ruidosa: 0.9118)")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Época 1/4 | Train Loss: 0.4338 | Val Accuracy: 0.8974
Época 2/4 | Train Loss: 0.1907 | Val Accuracy: 0.8994
Época 3/4 | Train Loss: 0.1518 | Val Accuracy: 0.9027
Época 4/4 | Train Loss: 0.1116 | Val Accuracy: 0.9009

Mejor Val Accuracy (Aug Limpia): 0.9027 (vs Sin Aug: 0.9074 | vs Aug Ruidosa: 0.9118)


In [ ]:
print("=== Reporte Global en Validación (xlm-roberta-large + Aumentación Limpia) ===")
print(classification_report(val_df['target'], val_df['pred_clean_aug'], target_names=['negativo', 'neutral', 'positivo'], digits=4))

sin_aug_acc = {'en': 0.8824, 'es': 0.9145, 'es-en': 0.6818, 'fr': 0.8931, 'other': 1.0000, 'pt': 0.9160}
aug1_acc    = {'en': 0.8824, 'es': 0.9189, 'es-en': 0.6364, 'fr': 0.9075, 'other': 1.0000, 'pt': 0.9187}

print("=== Tabla Comparativa por Idioma (Sin Aug vs Aug Ruidosa vs Aug Limpia) ===")
for lang, grp in val_df.groupby('lang'):
    acc_clean = accuracy_score(grp['target'], grp['pred_clean_aug'])
    acc_0 = sin_aug_acc.get(lang, 0.0)
    acc_1 = aug1_acc.get(lang, 0.0)
    print(f"Idioma {lang:6s} (n={len(grp):4d}): SinAug={acc_0:.4f} | Aug1={acc_1:.4f} | AugLimpia={acc_clean:.4f} (Delta vs SinAug={acc_clean - acc_0:+.4f})")

model.load_state_dict(torch.load('best_xlmr_large_clean_aug.pt'))
model.eval()
eval_probs = []

with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        eval_probs.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())

eval_probs = np.vstack(eval_probs)
np.save('eval_probs_xlmr_large_clean_aug.npy', eval_probs)

eval_preds = np.argmax(eval_probs, axis=1)
submission_3b = pd.DataFrame({
    'id': eval_df['id'],
    'answer': [id2label[p] for p in eval_preds]
})

submission_3b.to_csv('submission_3b_xlmr_clean_aug.csv', index=False)
print("\nArchivo generado: submission_3b_xlmr_clean_aug.csv")
print("Distribución de predicciones en eval (debería estar cerca de pos ~0.42, neg ~0.38, neu ~0.19):")
print(submission_3b['answer'].value_counts(normalize=True).round(4))

=== Reporte Global en Validación (xlm-roberta-large + Aumentación Limpia) ===
              precision    recall  f1-score   support

    negativo     0.9221    0.7992    0.8563      1230
     neutral     1.0000    0.9946    0.9973       932
    positivo     0.8244    0.9359    0.8766      1249

    accuracy                         0.9027      3411
   macro avg     0.9155    0.9099    0.9101      3411
weighted avg     0.9076    0.9027    0.9023      3411

=== Tabla Comparativa por Idioma (Sin Aug vs Aug Ruidosa vs Aug Limpia) ===
Idioma en     (n= 391): SinAug=0.8824 | Aug1=0.8824 | AugLimpia=0.8798 (Delta vs SinAug=-0.0026)
Idioma es     (n=2282): SinAug=0.9145 | Aug1=0.9189 | AugLimpia=0.9110 (Delta vs SinAug=-0.0035)
Idioma es-en  (n=  22): SinAug=0.6818 | Aug1=0.6364 | AugLimpia=0.5455 (Delta vs SinAug=-0.1363)
Idioma fr     (n= 346): SinAug=0.8931 | Aug1=0.9075 | AugLimpia=0.8844 (Delta vs SinAug=-0.0087)
Idioma other  (n=   1): SinAug=1.0000 | Aug1=1.0000 | AugLimpia=1.0000 (Delta

In [ ]:
# 1. Accuracy de validación re-ponderado según los pesos reales de idiomas en eval.csv
eval_lang_weights = {'es': 0.6065, 'en': 0.1367, 'pt': 0.1295, 'fr': 0.1172, 'es-en': 0.0100}
weighted_val_acc = 0.0
for lang, w in eval_lang_weights.items():
    sub = val_df[val_df['lang'] == lang]
    if len(sub) > 0:
        weighted_val_acc += w * accuracy_score(sub['target'], sub['pred_clean_aug'])

# 2. Accuracy específico en reseñas difíciles (concesiones / contraste)
hard_val = val_df[val_df['text'].str.lower().str.contains(r'\b(?:pero|aunque|sin embargo|but|however|mas|porém|mais)\b', regex=True)]
hard_acc = accuracy_score(hard_val['target'], hard_val['pred_clean_aug'])

print(f"Val Accuracy Estándar:       {accuracy_score(val_df['target'], val_df['pred_clean_aug']):.4f}")
print(f"Val Accuracy Ponderado Eval: {weighted_val_acc:.4f}")
print(f"Val Accuracy Casos Difíciles:{hard_acc:.4f} (n={len(hard_val)})")

Val Accuracy Estándar:       0.9027
Val Accuracy Ponderado Eval: 0.9002
Val Accuracy Casos Difíciles:0.8890 (n=766)


Aunque el puntaje en Kaggle subió frente a la Iteración 3 (0.89047 $\rightarrow$ 0.89194), siguió por debajo del modelo sin aumentación (0.89685), y el recall de negativo cayó de 0.8699 a 0.7992 (desplazando las predicciones en eval.csv hacia positivo con 47.46%). Esto demostró empíricamente que duplicar reseñas con concesiones mediante ediciones léxicas estáticas hace que un modelo de 560M parámetros memorice los adjetivos positivos iniciales, motivando el paso hacia estrategias de aumentación balanceadas o en el espacio latente (R-Drop / Back-Translation simétrico).